# Roadmap 2 — Document Augmentation for Q&A

This notebook builds on the **final Step 1 chatbot**. Use it like this:

| File | Purpose |
|------|---------|
| `step1.ipynb` | Learning notebook — experiments, keep as reference |
| `step2.ipynb` | Current working notebook — baseline + new Step 2 features |
| `config.py`, `pricing.py`, `state.py`, `chatbot.py` | Reusable logic — graduate stable code here |
| `app.py` | Runnable app — run with `python app.py` when the notebook version works |

**Workflow:** experiment in this notebook → move stable code into the Python modules → run `app.py` to verify.

## Step 1 Baseline — Import the working chatbot

The cells below load the final Step 1 product from the project modules.
You do **not** need to copy-paste code from `step1.ipynb` again.

In [48]:
import sys
from pathlib import Path
import tiktoken

# Allow imports from the project root (one level up from notebooks/)
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import gradio as gr

from config import OPENAI_API_KEY, MODEL_NAME
from chatbot import chat, clear_conversation, get_request_stats, get_session_stats
from state import create_initial_state
from helperFunctions import analyze_tokens, estimate_input_tokens

print("API key loaded:", bool(OPENAI_API_KEY))
print("Model:", MODEL_NAME)

API key loaded: True
Model: gpt-4.1-mini


In [49]:
import importlib
import helperFunctions

importlib.reload(helperFunctions)

from helperFunctions import analyze_tokens, estimate_input_tokens

## Step 2 — Load a document

Before the chatbot can answer questions about a document, load it into memory.

In [51]:
document_path = PROJECT_ROOT / "data" / "profile1.txt"

try:
    document_text = document_path.read_text(encoding="utf-8")
    print("Document loaded successfully.")
    print(f"Characters: {len(document_text)}")
except FileNotFoundError:
    document_text = ""
    print(f"Document not found at {document_path}")

Document loaded successfully.
Characters: 51707


In [52]:
stats = analyze_tokens(document_text,MODEL_NAME)
for key, value in stats.items():
    print(f"{key}: {value}")

characters: 51707
words: 6852
tokens: 8226
context_window: 1048576
context_percentage: 0.7844924926757812


In [57]:
def create_initial_state_with_document():
    return {
        "messages": [
            {
                "role": "system",
                "content": f"""You answer questions using the reference document below.
If the document does not contain enough information, say so.

{document_text}""",
            }
        ],
        "usage": {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "total_cost": 0,
        },
    }

state = create_initial_state_with_document()
user_message = "Who is John Doe?"


## Launch the Step 1 chatbot

In [58]:
with gr.Blocks() as demo:

    state = gr.State(create_initial_state_with_document())

    request_stats = gr.Markdown(get_request_stats())

    session_stats = gr.Markdown(
        get_session_stats(create_initial_state_with_document()["usage"])
    )

    chat_interface = gr.ChatInterface(
        fn=chat,
        title="Framework Free Chatbot",
        description="My first chatbot :p",
        additional_inputs=[state],
        additional_outputs=[state, request_stats, session_stats],
    )

    clear_btn = gr.Button("🗑️ Clear Conversation")

    clear_btn.click(
        clear_conversation,
        outputs=[
            chat_interface.chatbot,
            state,
            request_stats,
            session_stats,
        ],
    )

In [59]:
demo.launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [60]:
demo.close()

Closing server running on port: 7862
